<a href="https://colab.research.google.com/github/Ihsan-Fazal/LULC/blob/main/Statistical_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np

# Load saved predictions and ground-truth labels
y_true = np.load("y_true_hybrid.npy")
y_pred_hybrid = np.load("y_pred_hybrid.npy")
y_pred_deep = np.load("y_pred_deep.npy")

print("Ground truth shape:", y_true.shape)
print("Hybrid predictions shape:", y_pred_hybrid.shape)
print("Deep predictions shape:", y_pred_deep.shape)

Ground truth shape: (125,)
Hybrid predictions shape: (125,)
Deep predictions shape: (125,)


In [2]:
print("Ground truth classes:", np.unique(y_true))
print("Hybrid prediction classes:", np.unique(y_pred_hybrid))
print("Deep prediction classes:", np.unique(y_pred_deep))

Ground truth classes: [0 1 2 3 4 5]
Hybrid prediction classes: [0 1 2 3 4 5]
Deep prediction classes: [0 1 2 3 4 5]


In [3]:
from statsmodels.stats.contingency_tables import mcnemar
import numpy as np

# Correct/incorrect status for each of the same 125 test patches
hybrid_correct = (y_pred_hybrid == y_true)
deep_correct = (y_pred_deep == y_true)

# McNemar 2x2 contingency table
table = np.array([
    [
        np.sum(hybrid_correct & deep_correct),
        np.sum(hybrid_correct & ~deep_correct)
    ],
    [
        np.sum(~hybrid_correct & deep_correct),
        np.sum(~hybrid_correct & ~deep_correct)
    ]
])

print("McNemar contingency table:")
print(table)

# Exact McNemar test
result = mcnemar(table, exact=True)

print("\nMcNemar test:")
print("Statistic:", result.statistic)
print("p-value:", result.pvalue)

McNemar contingency table:
[[101   3]
 [ 11  10]]

McNemar test:
Statistic: 3.0
p-value: 0.057373046875


In [4]:
from sklearn.metrics import accuracy_score, f1_score, jaccard_score

print("Hybrid accuracy:",
      accuracy_score(y_true, y_pred_hybrid))

print("Deep accuracy:",
      accuracy_score(y_true, y_pred_deep))

print("Hybrid F1:",
      f1_score(y_true, y_pred_hybrid, average="macro"))

print("Deep F1:",
      f1_score(y_true, y_pred_deep, average="macro"))

print("Hybrid IoU:",
      jaccard_score(y_true, y_pred_hybrid, average="macro"))

print("Deep IoU:",
      jaccard_score(y_true, y_pred_deep, average="macro"))

Hybrid accuracy: 0.832
Deep accuracy: 0.896
Hybrid F1: 0.8044543325122292
Deep F1: 0.9032278778525753
Hybrid IoU: 0.6890592046842047
Deep IoU: 0.8341335623257665


In [5]:
from sklearn.metrics import accuracy_score, f1_score, jaccard_score
import numpy as np

def accuracy_metric(y_true, y_pred):
    return accuracy_score(y_true, y_pred)

def f1_metric(y_true, y_pred):
    return f1_score(
        y_true,
        y_pred,
        average="macro",
        zero_division=0
    )

def iou_metric(y_true, y_pred):
    return jaccard_score(
        y_true,
        y_pred,
        average="macro",
        zero_division=0
    )


def bootstrap_difference_ci(
    y_true,
    y_pred_hybrid,
    y_pred_deep,
    metric_func,
    n_bootstrap=10000,
    confidence=0.95,
    seed=20
):
    rng = np.random.default_rng(seed)
    n = len(y_true)

    differences = np.empty(n_bootstrap)

    for i in range(n_bootstrap):

        # Resample the SAME patches for both models
        indices = rng.integers(0, n, size=n)

        hybrid_score = metric_func(
            y_true[indices],
            y_pred_hybrid[indices]
        )

        deep_score = metric_func(
            y_true[indices],
            y_pred_deep[indices]
        )

        differences[i] = deep_score - hybrid_score

    alpha = 1 - confidence

    lower = np.percentile(
        differences,
        100 * (alpha / 2)
    )

    upper = np.percentile(
        differences,
        100 * (1 - alpha / 2)
    )

    mean_difference = np.mean(differences)

    return mean_difference, lower, upper

In [6]:
metrics = {
    "Accuracy": accuracy_metric,
    "Macro F1": f1_metric,
    "Mean IoU": iou_metric
}

for name, metric in metrics.items():

    difference, lower, upper = bootstrap_difference_ci(
        y_true,
        y_pred_hybrid,
        y_pred_deep,
        metric
    )

    print(f"\n{name}")
    print(f"Deep - Hybrid difference: {difference:.4f}")
    print(f"95% CI: [{lower:.4f}, {upper:.4f}]")


Accuracy
Deep - Hybrid difference: 0.0635
95% CI: [0.0080, 0.1200]

Macro F1
Deep - Hybrid difference: 0.1013
95% CI: [0.0290, 0.1798]

Mean IoU
Deep - Hybrid difference: 0.1441
95% CI: [0.0492, 0.2378]
